<a href="https://colab.research.google.com/github/Shruthi10357/GEN-AI-Practice/blob/main/Intro_to_Vector_Databases_ChromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Vector Databases (Installing and Running ChromaDB)**Level:** Scratch → Intermediate**Runs on:** Google Colab (Free CPU)### What you will learn1. What a Vector Database is and why we need one2. Installing ChromaDB (free, open-source, runs locally)3. Creating a Collection (like a table, but for vectors)4. Adding documents (ChromaDB auto-generates embeddings)5. Querying by meaning (semantic search)6. Using your own Hugging Face embeddings with ChromaDB (intermediate)7. Updating and deleting records> No API key, no cloud account, no cost. Everything runs in-memory inside this notebook.

## 1. Why a Vector Database?A normal database (SQL) searches by **exact match** — e.g., `WHERE name = 'dog'`.A **Vector Database** stores embeddings (numeric meaning-vectors) and searches by **similarity** — e.g., "find text that *means* something like 'dog'", even if the word "dog" never appears.| Normal Database | Vector Database ||---|---|| Matches exact text/values | Matches by meaning || Uses SQL queries | Uses similarity search || Good for structured data | Good for text, images, audio embeddings |**ChromaDB** is one of the simplest free, open-source vector databases — great for learning.

### 2. Install ChromaDBRun this once. `-q` keeps the output clean.

In [1]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

### 3. Create a ChromaDB ClientWe use an **in-memory / ephemeral client** — perfect for learning in Colab.(Data will reset when the runtime restarts. For permanent storage, `chromadb.PersistentClient(path="...")` is used instead — shown later.)

In [4]:
import chromadb

client = chromadb.Client()
print("ChromaDB client created successfully!")

ChromaDB client created successfully!


### 4. Create a CollectionA **Collection** in ChromaDB is like a table — it stores your documents, their embeddings, and metadata.

In [6]:
collection = client.create_collection(name="my_first_collection")
print("Collection created:", collection.name)

Collection created: my_first_collection


### 5. Add DocumentsWe simply give ChromaDB plain text. By default, it **automatically converts text into embeddings** for us using a built-in free model — no extra setup needed.

In [8]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers.",
        "Paris is the capital city of France."
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5", "doc6"]
)

print("Documents added! Total documents:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 10.1MiB/s]


Documents added! Total documents: 6


**Discussion Point:** Every document got an `id` (like a primary key). Behind the scenes, ChromaDB converted each sentence into an embedding vector and stored it — just like we did manually in the earlier embeddings notebook.

### 6. Query by Meaning (Semantic Search)Now we ask a question in natural language. ChromaDB converts our query into an embedding too, then finds the closest matching documents.

In [10]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f}  |  {doc}")

Distance: 0.9317  |  Deep learning uses neural networks with many layers.
Distance: 0.9516  |  Python is a popular programming language for AI.


**Discussion Point:** Lower `distance` = more similar (closer meaning). Notice the AI/ML related sentences come back first, even though the query never used the exact same words.

In [11]:
# Try another queryresults = collection.query(    query_texts=["What is the capital of France?"],    n_results=2)for doc, distance in zip(results["documents"][0], results["distances"][0]):    print(f"Distance: {distance:.4f}  |  {doc}")

### 7. Adding Metadata (Intermediate)Real-world documents usually have extra info (category, author, date). ChromaDB lets us store this alongside each document and filter on it.

In [12]:
collection.add(    documents=[        "The new smartphone has an amazing camera.",        "Stock prices rose sharply after the earnings report."    ],    metadatas=[        {"category": "technology"},        {"category": "finance"}    ],    ids=["doc7", "doc8"])# Query only within the "technology" categoryresults = collection.query(    query_texts=["Tell me about new gadgets"],    n_results=2,    where={"category": "technology"})for doc, distance in zip(results["documents"][0], results["distances"][0]):    print(f"Distance: {distance:.4f}  |  {doc}")

### 8. Using Your Own Hugging Face Embeddings (Intermediate)By default ChromaDB uses its own built-in embedding model. But we can plug in **any** Hugging Face model, such as `all-MiniLM-L6-v2` from the earlier notebook, for full control.

In [13]:
!pip install -q sentence-transformers

In [ ]:
from chromadb.utils import embedding_functionshf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(    model_name="all-MiniLM-L6-v2")# Create a new collection that uses our chosen Hugging Face modelcustom_collection = client.create_collection(    name="custom_embedding_collection",    embedding_function=hf_embedder)custom_collection.add(    documents=[        "I love dogs",        "I like puppies",        "The stock market crashed today"    ],    ids=["a1", "a2", "a3"])results = custom_collection.query(    query_texts=["puppy lovers"],    n_results=2)for doc, distance in zip(results["documents"][0], results["distances"][0]):    print(f"Distance: {distance:.4f}  |  {doc}")

### 9. Update and Delete Records

In [14]:
# Update a document's textcollection.update(    ids=["doc4"],    documents=["I enjoy playing football and basketball on weekends."])print("Updated doc4")# Delete a documentcollection.delete(ids=["doc2"])print("Deleted doc2")print("Remaining documents:", collection.count())

### 10. Persistent Storage (So Data Survives a Restart)The examples above used an in-memory client, which resets on restart. For real projects, use `PersistentClient` to save data to disk.

In [16]:
persistent_client = chromadb.PersistentClient(path="/content/chroma_db")
persistent_collection = persistent_client.get_or_create_collection(name="persistent_demo")

persistent_collection.add(
    documents=["This data will be saved to disk."],
    ids=["p1"]
)

print("Saved to disk at /content/chroma_db")
print("Count:", persistent_collection.count())

Saved to disk at /content/chroma_db
Count: 1


### 11. Key Takeaways (Recap for Students)| Concept | Meaning ||---|---|| Vector Database | Stores embeddings, searches by meaning/similarity || ChromaDB | Free, open-source, runs locally — no cloud needed || Collection | Like a table — holds documents + embeddings + metadata || `add()` | Insert documents (embeddings created automatically) || `query()` | Search by meaning, returns closest matches || `where={}` | Filter results using metadata || `PersistentClient` | Saves the database to disk instead of memory |### 12. Practice Exercises1. Add 5 of your own sentences to a new collection and query them with a related question.2. Add a `"category"` metadata field to every document, then filter a query by two different categories.3. Swap the embedding function to `'all-mpnet-base-v2'` and compare the distance scores to `all-MiniLM-L6-v2`.